In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score
)

N_TRIALS = 30
SUBSAMPLE_FRAC = 0.90
k_values = range(2, 36)
K_FINAL = 30

all_runs_metrics = []
ari_results = []

n = X_final.shape[0]

full_labels = {}

full_sil = []
full_db = []

for k in k_values:

    model_full = AgglomerativeClustering(
        n_clusters=k,
        linkage='ward'
    )

    labels_full_k = model_full.fit_predict(X_final)

    full_labels[k] = labels_full_k

    full_sil.append(
        silhouette_score(X_final, labels_full_k)
    )

    full_db.append(
        davies_bouldin_score(X_final, labels_full_k)
    )

df_full = pd.DataFrame({
    "k": list(k_values),
    "Silhouette_full": full_sil,
    "Davies_Bouldin_full": full_db
})

for trial in range(N_TRIALS):

    print(f" Trial {trial + 1} / {N_TRIALS}")

    sample_indices = np.random.choice(
        n,
        size=int(SUBSAMPLE_FRAC * n),
        replace=False
    )

    X_sub = X_final[sample_indices]

    sil_scores = []
    db_scores = []

    for k in k_values:

        model_sub = AgglomerativeClustering(
            n_clusters=k,
            linkage='ward'
        )

        labels_sub = model_sub.fit_predict(X_sub)

        sil_scores.append(
            silhouette_score(X_sub, labels_sub)
        )

        db_scores.append(
            davies_bouldin_score(X_sub, labels_sub)
        )

    df_trial = pd.DataFrame({
        "trial": trial + 1,
        "k": list(k_values),
        "Silhouette": sil_scores,
        "Davies_Bouldin": db_scores
    })

    all_runs_metrics.append(df_trial)

    for k in k_values:

        model_sub = AgglomerativeClustering(
            n_clusters=k,
            linkage='ward'
        )

        labels_sub = model_sub.fit_predict(X_sub)

        labels_full_subset = full_labels[k][sample_indices]

        ari = adjusted_rand_score(
            labels_full_subset,
            labels_sub
        )

        ari_results.append({
            "trial": trial + 1,
            "k": k,
            "ARI": ari
        })

df_all = pd.concat(
    all_runs_metrics,
    ignore_index=True
)

df_ari_trials = pd.DataFrame(ari_results)

df_k_stability = (
    df_ari_trials
    .groupby("k")["ARI"]
    .agg(
        ARI_mean="mean",
        ARI_std="std"
    )
    .reset_index()
)

df_k_stability["ARI_mean"] = df_k_stability["ARI_mean"].round(2)
df_k_stability["ARI_std"] = df_k_stability["ARI_std"].round(2)

df_stats = (
    df_all
    .groupby("k")
    .agg(
        Sil_mean=("Silhouette", "mean"),
        Sil_std=("Silhouette", "std"),
        DB_mean=("Davies_Bouldin", "mean"),
        DB_std=("Davies_Bouldin", "std")
    )
    .reset_index()
)

print("\nARI values:")
print(df_k_stability)

def plot_metric_pub(
    k_vals,
    full_vals,
    mean_vals,
    std_vals,
    ylabel,
    filename
):

    plt.figure(
        figsize=(12, 6),
        dpi=300
    )

    plt.plot(
        k_vals,
        full_vals,
        color="black",
        linewidth=3,
        label="Full Data (100%)"
    )

    plt.plot(
        k_vals,
        mean_vals,
        color="#0072B2",
        linewidth=2.5,
        label="90% Subsample Mean"
    )

    plt.fill_between(
        k_vals,
        mean_vals - std_vals,
        mean_vals + std_vals,
        color="#0072B2",
        alpha=0.25
    )

    plt.xlabel(
        "Number of Clusters (k)",
        fontsize=30,
        fontweight="bold"
    )

    plt.ylabel(
        ylabel,
        fontsize=30,
        fontweight="bold"
    )

    plt.xticks(
        fontsize=20,
        fontweight="bold"
    )

    plt.yticks(
        fontsize=20,
        fontweight="bold"
    )

    ax = plt.gca()

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    plt.grid(
        True,
        linestyle="--",
        alpha=0.25
    )

    plt.legend(
        fontsize=18,
        frameon=False
    )

    plt.tight_layout()

    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


k_vals = df_stats["k"].values

plot_metric_pub(
    k_vals,
    df_full["Silhouette_full"].values,
    df_stats["Sil_mean"].values,
    df_stats["Sil_std"].values,
    "Silhouette Score",
    "Robustness-check_silhouette_plot.png"
)

plot_metric_pub(
    k_vals,
    df_full["Davies_Bouldin_full"].values,
    df_stats["DB_mean"].values,
    df_stats["DB_std"].values,
    "Davies–Bouldin score",
    "Robustness-check_davies_bouldin_plot.png"
)

df_k_stability.to_csv(
    "ARI_values_table.csv",
    index=False
)